# 11 — ETL e Qualidade

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 11/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Materiais complementares

- [Guia Técnico](./reference_materials/11_etl_qualidade.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_etl_qualidade.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
Construir um pipeline idempotente e evitar alterar a fonte.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
Ingestão, validação, casting, deduplicação e trilhas de auditoria.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
import pandas as pd
v = pd.read_csv('datasets/flights.csv')
campos = ['carrier','origin','dest','distance']
assert set(campos).issubset(v.columns)
limpo = v.copy()
limpo['distance'] = pd.to_numeric(limpo['distance'], errors='coerce')
limpo = limpo.dropna(subset=['carrier','origin','dest'])
limpo = limpo[limpo['carrier'].astype(str).str.strip().ne('')]
assert limpo[campos[:3]].notna().all().all()
print('Original:',len(v),'| Registros válidos (campos obrigatórios):',len(limpo))
print('Nulos em distância mantidos para diagnosticar completude:',int(limpo.distance.isna().sum()))


Original: 10000 | Registros válidos (campos obrigatórios): 10000
Nulos em distância mantidos para diagnosticar completude: 0


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
print('Sem distância:',int(limpo['distance'].isna().sum()))
resumo=limpo.groupby('carrier',as_index=False)['distance'].agg(['count','mean'])
print(resumo.head())
print('Nenhum CSV original foi alterado.')

Sem distância: 0
  carrier  count         mean
0      AA    482  1739.230290
1      AS   3784  1218.056554
2      B6    214  1635.308411
3      DL   1082  1566.137708
4      F9    181  1033.723757
Nenhum CSV original foi alterado.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 11 reuniu conceitos e verificações de **contratos, duplicações e idempotência**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** ETL idempotente. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **ETL idempotente** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Idempotência: transformação sem mutar os dados de origem

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
import pandas as pd
raw = pd.DataFrame({'id':[1,2,2], 'quantidade':[3,4,4]})
def transformar(df):
    # Somente IDs representam eventos duplicados por contrato deste exemplo.
    assert not df.loc[df.duplicated('id',keep=False)].groupby('id')['quantidade'].nunique().gt(1).any(), 'IDs conflitantes'
    return df.drop_duplicates(subset=['id']).sort_values('id').reset_index(drop=True)
a = transformar(raw)
b = transformar(a)
assert a.equals(b)
assert len(raw)==3 and len(a)==2
print('Linhas:',len(raw),'->',len(a),'| operação idempotente:',a.equals(b))
print('Deduplicar por ID só é válido com regra de identidade explícita.')


Linhas: 3 -> 2 | operação idempotente: True
Deduplicar por ID só é válido com regra de identidade explícita.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **ETL idempotente** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
